# RSNA Knee: DINOv2 336px fold trainer V2

This notebook trains one fold and one seed per Kaggle run. It uses twelve slices from
six protocol-aware sequence slots, stores decoded studies in a compressed temporary
cache, fine-tunes DINOv2-small, and exports a checkpoint plus prediction artifacts.

Required Kaggle inputs:

- Competition: `rsna-knee-abnormality-detection`
- Model: `metaresearch/dinov2/PyTorch/small/1`
- Recommended labels: `pilkwang/rsna-knee-llm-labels`

Internet must remain disabled. Change `TRAIN_FOLD` and `RUN_SEED` between runs; do not
try to train a twenty-member ensemble in one nine-hour session.

In [ ]:
from __future__ import annotations

import gc
import hashlib
import math
import os
import random
import re
import shutil
import time
import unicodedata
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from sklearn.metrics import roc_auc_score

## Configuration

The defaults target a Kaggle T4 notebook and leave a margin under the nine-hour limit.
Set `DEBUG=True` only to verify the pipeline; its submission is not competitive.

In [ ]:
RUN_SEED = 17
TRAIN_FOLD = 0
DEBUG = False

TARGETS = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture",
]

# name, anatomical plane, fluid-sensitive flag, fat-suppression flag
SLOTS = [
    ("SAG_FLUID_FS", "Sagittal", 1, 1),
    ("COR_FLUID_FS", "Coronal", 1, 1),
    ("AX_FLUID_FS", "Axial", 1, 1),
    ("SAG_FLUID_NOFS", "Sagittal", 1, 0),
    ("COR_T1", "Coronal", 0, 0),
    ("SAG_T1", "Sagittal", 0, 0),
]
N_SLOTS = len(SLOTS)

IMAGE_SIZE = 336
GROUP_SIZE = 3
N_GROUPS = 4
CACHE_SLICES = GROUP_SIZE * N_GROUPS
CROP_MM = 130.0
PIXEL_THREADS = 12
CACHE_IO_THREADS = 4
CACHE_ROOT = Path("/kaggle/temp/rsna-knee-v2-cache")
if not Path("/kaggle").is_dir():
    CACHE_ROOT = Path("/tmp/rsna-knee-v2-cache")

N_FOLDS = 5
EPOCHS = 30
BATCH_STUDIES = 2
EVAL_BATCH = 2
UNFREEZE_LAST = 6
LR_BACKBONE = 1e-5
LR_HEAD = 5e-4
WEIGHT_DECAY = 0.05
RANK_LOSS_WEIGHT = 0.05
GOLD_WEIGHT = 4.0
TIME_BUDGET_SECONDS = 8.15 * 3600
TRAIN_STOP_SECONDS = 7.50 * 3600
SAVE_FOLD_MODELS = True
DELETE_TEMP_CACHE = True
REQUIRE_EXTERNAL_LABELS = True

if DEBUG:
    N_FOLDS = 2
    TRAIN_FOLD = 0
    EPOCHS = 1
    BATCH_STUDIES = 1
    EVAL_BATCH = 1
    N_GROUPS = 1
    CACHE_SLICES = GROUP_SIZE
    UNFREEZE_LAST = 1

START_TIME = time.time()


def log(message: str) -> None:
    elapsed = time.time() - START_TIME
    print(f"[{elapsed:7.1f}s] {message}", flush=True)


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


if not 0 <= TRAIN_FOLD < N_FOLDS:
    raise ValueError(f"TRAIN_FOLD must be in [0, {N_FOLDS - 1}], got {TRAIN_FOLD}")

seed_everything(RUN_SEED + TRAIN_FOLD)
torch.backends.cudnn.benchmark = True
log(f"torch={torch.__version__} cuda={torch.cuda.is_available()}")

## Locate offline inputs

In [ ]:
def find_competition_root() -> Path:
    candidates = [
        Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
        Path("/kaggle/input/rsna-knee-abnormality-detection"),
        Path("input"),
        Path("input_meta"),
        Path("."),
    ]
    for path in candidates:
        if (path / "train.csv").is_file() and (path / "test.csv").is_file():
            return path

    base = Path("/kaggle/input")
    if base.is_dir():
        for train_csv in base.glob("**/train.csv"):
            path = train_csv.parent
            if (path / "test.csv").is_file() and (path / "train_series.csv").is_file():
                return path
    raise FileNotFoundError("Could not locate the competition input directory")


def find_dinov2_root(variant: str = "small") -> Path:
    base = Path("/kaggle/input")
    hits: list[Path] = []
    if base.is_dir():
        direct_candidates = [
            base / "models/metaresearch/dinov2/pytorch/small/1",
            base / "dinov2/pytorch/small/1",
        ]
        hits.extend(path for path in direct_candidates if (path / "config.json").is_file())

        # Avoid recursively walking the very large competition DICOM directories.
        for root, directories, files in os.walk(base):
            directories[:] = [
                name for name in directories if name not in ("train_series", "test_series")
            ]
            if "config.json" in files and "dinov2" in root.lower():
                hits.append(Path(root))
    hits = list(dict.fromkeys(hits))
    preferred = [path for path in hits if variant.lower() in str(path).lower()]
    if preferred:
        return preferred[0]
    if hits:
        return hits[0]
    mounts = []
    if base.is_dir():
        mounts = sorted(path.name for path in base.iterdir() if path.is_dir())
    raise FileNotFoundError(
        "DINOv2 weights were not found. This is a Kaggle Model, not a Dataset. "
        "Open https://www.kaggle.com/models/metaresearch/dinov2 and attach "
        "PyTorch / small / version 1 to this notebook. "
        f"Visible /kaggle/input mounts: {mounts}"
    )


ROOT = find_competition_root()
log(f"competition root: {ROOT}")

# Fail before the expensive DICOM cache pass when the offline model input is missing.
DINO_ROOT = None
if Path("/kaggle/input").is_dir() or not DEBUG:
    DINO_ROOT = find_dinov2_root("small")
    log(f"DINOv2 root: {DINO_ROOT}")

train_df = pd.read_csv(ROOT / "train.csv")
test_df = pd.read_csv(ROOT / "test.csv")
train_series_df = pd.read_csv(ROOT / "train_series.csv")
test_series_df = pd.read_csv(ROOT / "test_series.csv")

if DEBUG:
    train_df = train_df.iloc[:96].copy()
    train_series_df = train_series_df[
        train_series_df["StudyInstanceUID"].isin(train_df["StudyInstanceUID"])
    ].copy()

log(f"train={train_df.shape} train_series={train_series_df.shape} test={test_df.shape}")


def write_fallback_submission() -> None:
    submission = test_df[["StudyInstanceUID"]].copy()
    for target in TARGETS:
        submission[target] = 0.5
    submission.to_csv("submission.csv", index=False)


write_fallback_submission()
log("wrote fallback submission.csv")

## Report-derived supervision

A mounted `report_labels*.csv` is preferred. The fallback below is intentionally
conservative: silence receives little weight, explicit negation receives medium weight,
and a named positive finding receives high weight. The 58 expert-labelled studies always
override report labels.

In [ ]:
_TRANSLATE = str.maketrans(
    {
        "\u0131": "i", "\u0130": "i", "\u015f": "s", "\u015e": "s",
        "\u011f": "g", "\u011e": "g", "\u00fc": "u", "\u00dc": "u",
        "\u00f6": "o", "\u00d6": "o", "\u00e7": "c", "\u00c7": "c",
        "\u00df": "ss",
    }
)


def normalize_report(text: object) -> str:
    if not isinstance(text, str):
        return ""
    text = text.translate(_TRANSLATE)
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.lower()
    text = re.sub(r"[^a-z0-9'./;:,+-]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


NEGATION = re.compile(
    r"\b(no|not|without|absent|absence|negative|normal|intact|preserved|"
    r"unremarkable|free of|geen|zonder|kein|keine|ohne|sin|ausencia|"
    r"pas de|sans|assenza|senza|yok|izlenmedi|saptanmadi|gorulmedi|"
    r"mevcut degil|aucun|aucune)\b"
)
TEAR = re.compile(
    r"\b(tear|torn|ruptur|rotura|scheur|riss|dechir|yirtik|kopma|sprain|desgarro|lesion)\w*\b"
)
MENISCUS = re.compile(r"\b(menisc|menisk)\w*\b")
MEDIAL = re.compile(r"\b(medial|mediale|interno|internal|inner|binnen|innen|ic)\b")
LATERAL = re.compile(r"\b(lateral|laterale|externo|external|outer|buiten|aussen|dis)\b")
OA_WORD = re.compile(
    r"\b(osteoarth|arthros|artros|gonarthros|chondr|condropat|cartilage|"
    r"kraakbeen|knorpel|degenerat)\w*\b"
)
PF_WORD = re.compile(
    r"\b(patellofemoral|patello femoral|femoropatell|retropatell|patella|rotul|trochlea)\w*\b"
)

ANCHORS = {
    "ACL": re.compile(
        r"\b(acl|anterior cruciate|cruzado anterior|lca|voorste kruisband|vorderes kreuzband|on capraz)\b"
    ),
    "MCL": re.compile(
        r"\b(mcl|medial collateral|colateral medial|colateral interno|innenband|ic yan bag)\b"
    ),
    "Effusion": re.compile(
        r"\b(effusion|joint fluid|derrame|gewrichtsvocht|erguss|epanchement|versamento|efuzyon)\w*\b"
    ),
    "Synovitis": re.compile(r"\b(synovit|sinovit|synovial hypertrophy|synovial proliferation)\w*\b"),
    "Baker's": re.compile(r"\b(baker|popliteal cyst|quiste popliteo|bakercyste|bakerzyste)\w*\b"),
    "Contusion": re.compile(
        r"\b(contusion|bone bruise|bone marrow edema|marrow edema|knochenmarkodem|kemik iligi odem)\w*\b"
    ),
    "Fracture": re.compile(r"\b(fractur|fractuur|fraktur|frattur|kirik|avulsion)\w*\b"),
}


def report_clauses(text: str) -> list[str]:
    return [part.strip() for part in re.split(r"[.;:\n\r]+", text) if part.strip()]


def nearby(clause: str, pattern: re.Pattern[str], start: int, end: int, window: int = 120) -> bool:
    region = clause[max(0, start - window) : min(len(clause), end + window)]
    return bool(pattern.search(region))


def negated(clause: str, start: int, end: int) -> bool:
    region = clause[max(0, start - 90) : min(len(clause), end + 90)]
    return bool(NEGATION.search(region))


def grade_positive(clause: str) -> float:
    if re.search(r"\b(trace|minimal|tiny|slight|grade 1|grade i)\b", clause):
        return 0.62
    if re.search(r"\b(marked|severe|large|complete|grade 3|grade iii)\b", clause):
        return 0.96
    return 0.86


def fallback_report_labels(report: object) -> dict[str, float]:
    clauses = report_clauses(normalize_report(report))
    result: dict[str, float] = {}
    for target in TARGETS:
        score, confidence = 0.35, 0.12
        explicit_negative = False
        for clause in clauses:
            candidates: list[re.Match[str]] = []
            positive = False
            if target in ("ACL", "MCL"):
                candidates = list(ANCHORS[target].finditer(clause))
                positive = bool(candidates and TEAR.search(clause))
            elif target in ("Medial Meniscus", "Lateral Meniscus"):
                side = MEDIAL if target.startswith("Medial") else LATERAL
                candidates = list(MENISCUS.finditer(clause))
                positive = any(
                    nearby(clause, side, match.start(), match.end())
                    and nearby(clause, TEAR, match.start(), match.end())
                    for match in candidates
                )
            elif target in ("Medial OA", "Lateral OA", "PF OA"):
                side = MEDIAL if target == "Medial OA" else LATERAL if target == "Lateral OA" else PF_WORD
                candidates = list(OA_WORD.finditer(clause))
                positive = any(nearby(clause, side, match.start(), match.end()) for match in candidates)
            else:
                candidates = list(ANCHORS[target].finditer(clause))
                positive = bool(candidates)
            for match in candidates:
                if negated(clause, match.start(), match.end()):
                    explicit_negative = True
            if positive and not any(negated(clause, m.start(), m.end()) for m in candidates):
                score = max(score, grade_positive(clause))
                confidence = 0.88
        if score == 0.35 and explicit_negative:
            score, confidence = 0.08, 0.68
        result[target] = score
        result[target + "__conf"] = confidence
    return result


def fallback_label_frame(frame: pd.DataFrame) -> pd.DataFrame:
    labels = pd.DataFrame([fallback_report_labels(report) for report in frame["Report"].fillna("")])
    labels.insert(0, "StudyInstanceUID", frame["StudyInstanceUID"].values)
    return labels.set_index("StudyInstanceUID")


def find_report_label_table() -> Path | None:
    base = Path("/kaggle/input")
    if not base.is_dir():
        return None
    candidates = list(base.glob("**/report_labels*.csv"))
    candidates += list(base.glob("**/*weak*labels*.csv"))
    for path in candidates:
        try:
            columns = pd.read_csv(path, nrows=1).columns
        except Exception:
            continue
        if "StudyInstanceUID" in columns and all(target in columns for target in TARGETS):
            return path
    return None


def build_report_labels(frame: pd.DataFrame) -> tuple[pd.DataFrame, str]:
    fallback = fallback_label_frame(frame)
    source = find_report_label_table()
    if source is None:
        return fallback, "multilingual rules fallback"
    external = pd.read_csv(source).drop_duplicates("StudyInstanceUID").set_index("StudyInstanceUID")
    overlap = fallback.index.intersection(external.index)
    if len(overlap) == 0:
        raise RuntimeError(f"Report-label table {source} has no matching StudyInstanceUID values")
    for target in TARGETS:
        values = pd.to_numeric(external.loc[overlap, target], errors="coerce").clip(0, 1)
        verdict_column = target + "__verdict"
        unknown = values.isna()
        if verdict_column in external.columns:
            unknown |= (
                external.loc[overlap, verdict_column].astype(str).str.upper().eq("UNK")
            )
        # Unknown is not negative. A neutral target with zero confidence contributes no
        # gradient once the confidence weights are constructed below.
        fallback.loc[overlap, target] = values.where(~unknown, 0.5).fillna(0.5)
        confidence_column = target + "__conf"
        if confidence_column in external.columns:
            confidence = pd.to_numeric(
                external.loc[overlap, confidence_column], errors="coerce"
            ).clip(0, 1)
            confidence = confidence.fillna(0.0).where(~unknown, 0.0)
            fallback.loc[overlap, confidence_column] = confidence
        else:
            fallback.loc[overlap, confidence_column] = np.where(~unknown, 0.85, 0.0)

    # Report silence is particularly common for synovitis. Use an addressed effusion
    # field only for genuinely missing synovitis labels, and keep its confidence modest.
    syn = pd.to_numeric(external.loc[overlap, "Synovitis"], errors="coerce")
    eff = pd.to_numeric(external.loc[overlap, "Effusion"], errors="coerce").clip(0, 1)
    syn_unknown = syn.isna()
    eff_unknown = eff.isna()
    if "Synovitis__verdict" in external.columns:
        syn_unknown |= (
            external.loc[overlap, "Synovitis__verdict"].astype(str).str.upper().eq("UNK")
        )
    if "Effusion__verdict" in external.columns:
        eff_unknown |= (
            external.loc[overlap, "Effusion__verdict"].astype(str).str.upper().eq("UNK")
        )
    use_effusion = syn_unknown & ~eff_unknown
    if use_effusion.any():
        studies = use_effusion.index[use_effusion]
        fallback.loc[studies, "Synovitis"] = eff.loc[studies]
        eff_conf_column = "Effusion__conf"
        if eff_conf_column in external.columns:
            eff_conf = pd.to_numeric(
                external.loc[studies, eff_conf_column], errors="coerce"
            ).fillna(0.0).clip(0, 1)
        else:
            eff_conf = pd.Series(0.85, index=studies)
        fallback.loc[studies, "Synovitis__conf"] = 0.5 * eff_conf
    return fallback, str(source)


report_labels, label_source = build_report_labels(train_df)
log(f"label source: {label_source}")
if REQUIRE_EXTERNAL_LABELS and label_source == "multilingual rules fallback" and not DEBUG:
    raise FileNotFoundError(
        "Competitive training requires the report-label table. Attach the Kaggle Dataset "
        "pilkwang/rsna-knee-llm-labels and rerun the notebook."
    )
gold = train_df.set_index("StudyInstanceUID")[TARGETS]
gold = gold[gold.notna().all(axis=1)]
log(f"gold studies: {len(gold)}")

label_diagnostics = []
for target in TARGETS:
    y = gold[target].astype(int).to_numpy()
    p = report_labels.loc[gold.index, target].astype(float).to_numpy()
    auc = roc_auc_score(y, p) if len(np.unique(y)) == 2 else np.nan
    label_diagnostics.append({"target": target, "gold_auc": auc})
display(pd.DataFrame(label_diagnostics).round(3))

## DICOM ordering and multi-plane cache

File names are SOP UIDs and are not slice order. Each selected series is sorted using
`ImageOrientationPatient` and `ImagePositionPatient`, then sampled from the central 60%.
Images are cropped to a 130 mm field of view and normalized by robust percentiles. The
cache is compressed on temporary disk; the equivalent dense RAM array is too large at
336 px x 12 slices.

In [ ]:
SLOT_TO_INDEX = {
    (plane, int(fluid), int(fat_suppression)): index
    for index, (_, plane, fluid, fat_suppression) in enumerate(SLOTS)
}

FATSAT_OPTIONS = {"FS", "FATSAT", "FAT_SAT", "FSAT"}
FATSAT_PATTERN = re.compile(
    r"\bfs\b|fatsat|fat sat|\bstir\b|\bspair\b|\bspir\b|water excit|"
    r"\btirm\b|\bfatsup\b"
)
T1_PATTERN = re.compile(r"\bt1\b|\bt1w\b")
T2_PATTERN = re.compile(r"\bt2\b|\bt2w\b")
PD_PATTERN = re.compile(r"\bpd\b|\bpdw\b|proton|\bdp\b|dens")


def series_protocol(row: object, path: Path, files: list[Path]) -> tuple[int, int]:
    """Recover fluid weighting and fat suppression from one representative header."""
    fluid = int(getattr(row, "Fluid_Sensitive", 0) or 0)
    fat_suppression = int(getattr(row, "Fat_Suppression", 0) or 0)
    if not files:
        return fluid, fat_suppression
    try:
        ds = pydicom.dcmread(str(files[len(files) // 2]), stop_before_pixels=True, force=True)
    except Exception:
        return fluid, fat_suppression

    description = " ".join(
        str(getattr(ds, name, "") or "") for name in ("SeriesDescription", "SequenceName")
    ).lower()
    description = re.sub(r"[_\-.]+", " ", description)
    raw_scan_options = getattr(ds, "ScanOptions", None)
    scan_options = set()
    if isinstance(raw_scan_options, str):
        scan_options = {
            token.strip().upper()
            for token in re.split(r"[\\,| ]+", raw_scan_options)
            if token.strip()
        }
    elif raw_scan_options is not None:
        scan_options = {str(value).strip().upper() for value in raw_scan_options}
    if FATSAT_PATTERN.search(description) or scan_options.intersection(FATSAT_OPTIONS):
        fat_suppression = 1

    repetition = pd.to_numeric(getattr(ds, "RepetitionTime", np.nan), errors="coerce")
    echo = pd.to_numeric(getattr(ds, "EchoTime", np.nan), errors="coerce")
    named_t1 = bool(T1_PATTERN.search(description))
    named_t2 = bool(T2_PATTERN.search(description))
    named_pd = bool(PD_PATTERN.search(description))
    if named_t1 and not named_t2 and not named_pd:
        fluid = 0
    elif named_t2 or named_pd:
        fluid = 1
    elif np.isfinite(repetition) and repetition < 800:
        fluid = 0
    elif (np.isfinite(echo) and echo > 60) or (
        np.isfinite(repetition) and repetition >= 800
    ):
        fluid = 1
    return fluid, fat_suppression


def select_series(frame: pd.DataFrame, split: str) -> dict[str, dict[int, Path]]:
    """Choose the thickest matching series for each recovered protocol slot."""
    jobs = list(frame.itertuples(index=False))

    def inspect(row: object):
        study = str(row.StudyInstanceUID)
        series = str(row.SeriesInstanceUID)
        path = ROOT / split / study / series
        try:
            files = sorted(
                (Path(entry.path) for entry in os.scandir(path) if entry.name.lower().endswith(".dcm")),
                key=lambda item: item.name,
            )
        except FileNotFoundError:
            files = []
        fluid, fat_suppression = series_protocol(row, path, files)
        key = SLOT_TO_INDEX.get((str(row.Anatomical_Plane), fluid, fat_suppression))
        return study, key, path, len(files)

    selected: dict[str, dict[int, tuple[Path, int]]] = {}
    with ThreadPoolExecutor(max_workers=PIXEL_THREADS) as pool:
        for study, key, path, count in pool.map(inspect, jobs):
            if key is None or count == 0:
                continue
            previous = selected.setdefault(study, {}).get(key)
            if previous is None or count > previous[1]:
                selected[study][key] = (path, count)
    return {
        study: {slot: value[0] for slot, value in slots.items()}
        for study, slots in selected.items()
    }


def geometry_scalar(ds: pydicom.dataset.FileDataset) -> float | None:
    try:
        orientation = np.asarray(ds.ImageOrientationPatient, dtype=np.float64)
        position = np.asarray(ds.ImagePositionPatient, dtype=np.float64)
        normal = np.cross(orientation[:3], orientation[3:])
        return float(np.dot(position, normal))
    except Exception:
        return None


def ordered_series_files(path: Path) -> tuple[list[Path], pydicom.dataset.FileDataset | None]:
    records = []
    first_header = None
    for file_path in path.glob("*.dcm"):
        try:
            ds = pydicom.dcmread(str(file_path), stop_before_pixels=True, force=True)
            if first_header is None:
                first_header = ds
            records.append((file_path, geometry_scalar(ds), int(getattr(ds, "InstanceNumber", 0) or 0)))
        except Exception:
            records.append((file_path, None, 0))
    geometry_count = sum(scalar is not None for _, scalar, _ in records)
    if geometry_count >= 2:
        records.sort(key=lambda item: (item[1] is None, item[1] or 0.0, item[0].name))
    else:
        records.sort(key=lambda item: (item[2], item[0].name))
    files = [item[0] for item in records]
    if files:
        try:
            first_header = pydicom.dcmread(
                str(files[len(files) // 2]), stop_before_pixels=True, force=True
            )
        except Exception:
            pass
    return files, first_header


def central_indices(n_files: int, count: int) -> np.ndarray:
    if n_files <= 1:
        return np.zeros(count, dtype=int)
    low = int(0.20 * (n_files - 1))
    high = int(0.80 * (n_files - 1))
    return np.linspace(low, max(low, high), count).astype(int)


def center_crop_physical(image: np.ndarray, spacing: object) -> np.ndarray:
    try:
        spacing_values = np.asarray(spacing, dtype=np.float32)
        mm_per_pixel = float(np.mean(spacing_values[:2]))
    except Exception:
        return image
    if not np.isfinite(mm_per_pixel) or mm_per_pixel <= 0:
        return image
    crop = int(round(CROP_MM / mm_per_pixel))
    height, width = image.shape[-2:]
    if crop < 32 or crop >= min(height, width):
        return image
    top = (height - crop) // 2
    left = (width - crop) // 2
    return image[top : top + crop, left : left + crop]


def header_laterality(header: pydicom.dataset.FileDataset | None) -> str:
    if header is None:
        return ""
    tagged = str(
        getattr(header, "ImageLaterality", None) or getattr(header, "Laterality", "")
    ).strip().upper()[:1]
    if tagged in ("L", "R"):
        return tagged
    try:
        orientation = np.asarray(header.ImageOrientationPatient, dtype=np.float64)
        position = np.asarray(header.ImagePositionPatient, dtype=np.float64)
        spacing = np.asarray(header.PixelSpacing, dtype=np.float64)
        rows = float(header.Rows)
        columns = float(header.Columns)
        center = (
            position[:3]
            + orientation[:3] * spacing[1] * columns / 2
            + orientation[3:6] * spacing[0] * rows / 2
        )
        if abs(float(center[0])) >= 20.0:
            return "R" if center[0] < 0 else "L"
    except Exception:
        pass
    return ""


def read_series(path: Path, plane: str) -> np.ndarray | None:
    files, header = ordered_series_files(path)
    if not files:
        return None
    arrays: list[np.ndarray | None] = []
    for index in central_indices(len(files), CACHE_SLICES):
        try:
            ds = pydicom.dcmread(str(files[int(index)]), force=True)
            image = ds.pixel_array.astype(np.float32)
            image = image * float(getattr(ds, "RescaleSlope", 1.0) or 1.0)
            image = image + float(getattr(ds, "RescaleIntercept", 0.0) or 0.0)
            if str(getattr(ds, "PhotometricInterpretation", "")) == "MONOCHROME1":
                image = image.max() - image
            image = center_crop_physical(image, getattr(ds, "PixelSpacing", None))
            tensor = torch.from_numpy(np.ascontiguousarray(image))[None, None]
            resized = F.interpolate(
                tensor, size=(IMAGE_SIZE, IMAGE_SIZE), mode="bilinear", align_corners=False
            )[0, 0].numpy()
            arrays.append(resized)
        except Exception:
            arrays.append(None)
    valid_indices = [index for index, array in enumerate(arrays) if array is not None]
    if not valid_indices:
        return None
    for index, array in enumerate(arrays):
        if array is None:
            nearest = min(valid_indices, key=lambda valid_index: abs(valid_index - index))
            arrays[index] = arrays[nearest]
    valid = [array for array in arrays if array is not None]
    values = np.concatenate([array.reshape(-1) for array in valid])
    low, high = np.percentile(values, [1, 99])
    scale = max(float(high - low), 1e-6)
    volume = np.stack(arrays)
    volume = np.clip((volume - low) / scale, 0, 1)
    laterality = header_laterality(header)
    if laterality == "R":
        if plane in ("Coronal", "Axial"):
            volume = volume[..., ::-1]
        elif plane == "Sagittal":
            volume = volume[::-1]
    return np.ascontiguousarray((volume * 255).round().astype(np.uint8))


class CompressedStudyCache:
    def __init__(self, root: Path, studies: list[str]):
        self.root = root
        self.studies = studies

    def path(self, index: int, group: int) -> Path:
        return self.root / f"{index:05d}_g{group}.npz"

    def load_group(self, indices: np.ndarray, group: int) -> np.ndarray:
        def load(index: int) -> np.ndarray:
            with np.load(self.path(int(index), group), allow_pickle=False) as item:
                return item["images"]

        with ThreadPoolExecutor(max_workers=min(CACHE_IO_THREADS, max(len(indices), 1))) as pool:
            rows = list(pool.map(load, indices.tolist()))
        return np.stack(rows) if rows else np.zeros(
            (0, N_SLOTS, GROUP_SIZE, IMAGE_SIZE, IMAGE_SIZE), dtype=np.uint8
        )


def build_cache(
    studies: list[str], selected: dict[str, dict[int, Path]], tag: str
) -> tuple[CompressedStudyCache, np.ndarray]:
    root = CACHE_ROOT / (
        f"{tag}_{IMAGE_SIZE}px_{CACHE_SLICES}sl_{int(CROP_MM)}mm_"
        + "_".join(slot[0] for slot in SLOTS)
    )
    root.mkdir(parents=True, exist_ok=True)
    mask = np.zeros((len(studies), N_SLOTS), dtype=np.float32)
    dense_gb = len(studies) * N_SLOTS * CACHE_SLICES * IMAGE_SIZE**2 / 1024**3
    free_gb = shutil.disk_usage(root).free / 1024**3
    log(
        f"{tag}: compressed cache for {len(studies)} studies; "
        f"dense equivalent={dense_gb:.1f} GB, free temporary disk={free_gb:.1f} GB"
    )

    def cache_one(item: tuple[int, str]) -> tuple[int, np.ndarray]:
        index, study = item
        paths = [root / f"{index:05d}_g{group}.npz" for group in range(N_GROUPS)]
        if all(path.is_file() for path in paths):
            with np.load(paths[0], allow_pickle=False) as saved:
                return index, saved["mask"].astype(np.float32)

        images = np.zeros(
            (N_SLOTS, CACHE_SLICES, IMAGE_SIZE, IMAGE_SIZE), dtype=np.uint8
        )
        study_mask = np.zeros(N_SLOTS, dtype=np.float32)
        for slot_index, path in selected.get(study, {}).items():
            plane = SLOTS[slot_index][1]
            volume = read_series(path, plane)
            if volume is not None:
                images[slot_index] = volume
                study_mask[slot_index] = 1.0
        for group, path in enumerate(paths):
            start = group * GROUP_SIZE
            temporary = path.with_suffix(".tmp")
            with open(temporary, "wb") as handle:
                np.savez_compressed(
                    handle,
                    images=images[:, start : start + GROUP_SIZE],
                    mask=study_mask,
                )
            os.replace(temporary, path)
        return index, study_mask

    jobs = list(enumerate(studies))
    with ThreadPoolExecutor(max_workers=CACHE_IO_THREADS) as pool:
        for completed, (index, study_mask) in enumerate(pool.map(cache_one, jobs), 1):
            mask[index] = study_mask
            if completed % 100 == 0 or completed == len(jobs):
                log(f"{tag}: cached {completed}/{len(jobs)} studies")
    return CompressedStudyCache(root, studies), mask


train_studies = train_df["StudyInstanceUID"].astype(str).tolist()
test_studies = test_df["StudyInstanceUID"].astype(str).tolist()
train_selected = select_series(train_series_df, "train_series")
test_selected = select_series(test_series_df, "test_series")
train_cache, train_mask = build_cache(train_studies, train_selected, "train")
test_cache, test_mask = build_cache(test_studies, test_selected, "test")
log(
    f"slot coverage train={train_mask.mean():.1%}, test={test_mask.mean():.1%}; "
    f"empty studies train={(train_mask.sum(1) == 0).sum()}"
)

## DINOv2 study model

Each non-empty sequence slot becomes one three-slice RGB-like input. DINOv2 encodes the
slots; each abnormality has its own learned attention query over the six sequence slots.
Empty slots are excluded from both the encoder and the attention softmax.

In [ ]:
class DiagnosisSlotHead(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 256, dropout: float = 0.2):
        super().__init__()
        self.projection = nn.Sequential(
            nn.LayerNorm(input_dim), nn.Linear(input_dim, hidden_dim), nn.GELU()
        )
        self.slot_embedding = nn.Parameter(torch.randn(N_SLOTS, hidden_dim) * 0.02)
        self.queries = nn.Parameter(torch.randn(len(TARGETS), hidden_dim) * 0.02)
        self.output_weight = nn.Parameter(torch.randn(len(TARGETS), hidden_dim) * 0.02)
        self.output_bias = nn.Parameter(torch.zeros(len(TARGETS)))
        self.dropout = nn.Dropout(dropout)
        self.scale = hidden_dim**-0.5

    def forward(self, features: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        hidden = self.projection(features) + self.slot_embedding[None]
        attention = torch.einsum("bsh,th->bts", hidden, self.queries) * self.scale
        attention = attention.masked_fill(mask[:, None] < 0.5, -1e4).softmax(dim=-1)
        context = self.dropout(torch.einsum("bts,bsh->bth", attention, hidden))
        return (context * self.output_weight[None]).sum(dim=-1) + self.output_bias


class KneeDINOv2(nn.Module):
    def __init__(self, backbone: nn.Module):
        super().__init__()
        self.backbone = backbone
        self.head = DiagnosisSlotHead(int(backbone.config.hidden_size) * 2)
        self.register_buffer(
            "image_mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
        )
        self.register_buffer(
            "image_std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
        )

    def forward(self, images: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        batch, slots = images.shape[:2]
        flat = images.reshape(batch * slots, *images.shape[2:]).float().div(255.0)
        valid = mask.reshape(-1) > 0.5
        features = torch.zeros(
            (batch * slots, self.backbone.config.hidden_size * 2),
            device=images.device,
            dtype=torch.float32,
        )
        if valid.any():
            normalized = (flat[valid] - self.image_mean) / self.image_std
            output = self.backbone(pixel_values=normalized).last_hidden_state
            encoded = torch.cat([output[:, 0], output[:, 1:].mean(dim=1)], dim=1)
            features[valid] = encoded.float()
        return self.head(features.reshape(batch, slots, -1), mask)


def build_model() -> KneeDINOv2:
    from transformers import AutoModel

    source = DINO_ROOT or find_dinov2_root("small")
    backbone = AutoModel.from_pretrained(str(source), local_files_only=True)
    for parameter in backbone.parameters():
        parameter.requires_grad = False
    layers = backbone.encoder.layer
    for layer in layers[max(0, len(layers) - UNFREEZE_LAST) :]:
        for parameter in layer.parameters():
            parameter.requires_grad = True
    for parameter in backbone.layernorm.parameters():
        parameter.requires_grad = True
    trainable = sum(parameter.numel() for parameter in backbone.parameters() if parameter.requires_grad)
    log(f"DINOv2 source={source}; trainable backbone parameters={trainable / 1e6:.1f}M")
    return KneeDINOv2(backbone)


def take_group(
    cache: CompressedStudyCache, indices: np.ndarray, group: int
) -> torch.Tensor:
    return torch.from_numpy(cache.load_group(indices, group))


def augment(images: torch.Tensor) -> torch.Tensor:
    batch, slots, channels, height, width = images.shape
    flat = images.float().reshape(batch * slots, channels, height, width)
    angles = (torch.rand(batch, device=images.device) * 2 - 1) * math.radians(7.0)
    scales = 1.0 + (torch.rand(batch, device=images.device) * 2 - 1) * 0.07
    shifts = (torch.rand(batch, 2, device=images.device) * 2 - 1) * 0.04
    cosine = torch.cos(angles) / scales
    sine = torch.sin(angles) / scales
    theta = torch.zeros(batch, 2, 3, device=images.device)
    theta[:, 0, 0], theta[:, 0, 1] = cosine, -sine
    theta[:, 1, 0], theta[:, 1, 1] = sine, cosine
    theta[:, :, 2] = shifts
    theta = theta.repeat_interleave(slots, dim=0)
    grid = F.affine_grid(theta, flat.shape, align_corners=False)
    flat = F.grid_sample(flat, grid, mode="bilinear", padding_mode="zeros", align_corners=False)
    intensity = 1.0 + (torch.rand(batch * slots, 1, 1, 1, device=images.device) * 2 - 1) * 0.10
    flat = (flat * intensity).clamp(0, 255)
    return flat.reshape(batch, slots, channels, height, width).round().to(torch.uint8)


def pairwise_rank_loss(logits: torch.Tensor, labels: torch.Tensor, weights: torch.Tensor) -> torch.Tensor:
    losses = []
    for target_index in range(logits.shape[1]):
        confident = weights[:, target_index] >= 0.6
        positives = logits[(labels[:, target_index] >= 0.6) & confident, target_index]
        negatives = logits[(labels[:, target_index] <= 0.4) & confident, target_index]
        if len(positives) and len(negatives):
            losses.append(F.softplus(-(positives[:, None] - negatives[None, :])).mean())
    return torch.stack(losses).mean() if losses else logits.new_tensor(0.0)


def macro_auc(labels: np.ndarray, predictions: np.ndarray) -> float:
    scores = []
    for index in range(labels.shape[1]):
        if len(np.unique(labels[:, index])) == 2:
            scores.append(roc_auc_score(labels[:, index], predictions[:, index]))
    return float(np.mean(scores)) if scores else float("nan")


@torch.inference_mode()
def predict(
    model: KneeDINOv2,
    cache: CompressedStudyCache,
    mask: np.ndarray,
    indices: np.ndarray,
    device: torch.device,
) -> np.ndarray:
    model.eval()
    predictions = []
    fracture_index = TARGETS.index("Fracture")
    for offset in range(0, len(indices), EVAL_BATCH):
        selection = indices[offset : offset + EVAL_BATCH]
        batch_mask = torch.from_numpy(mask[selection]).to(device)
        group_probabilities = []
        for group in range(N_GROUPS):
            images = take_group(cache, selection, group).to(device)
            with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
                group_logits = model(images, batch_mask).float()
            group_probabilities.append(torch.sigmoid(group_logits))
        stacked = torch.stack(group_probabilities, dim=1)
        pooled = stacked.mean(dim=1)
        pooled[:, fracture_index] = stacked[:, :, fracture_index].max(dim=1).values
        predictions.append(pooled.cpu().numpy())
    if not predictions:
        return np.zeros((0, len(TARGETS)), dtype=np.float32)
    return np.concatenate(predictions)

## Targets and report-grouped folds

Duplicate reports remain in one fold. This prevents a templated report from creating the
same weak target in both training and validation.

In [ ]:
Y = np.zeros((len(train_studies), len(TARGETS)), dtype=np.float32)
W = np.zeros_like(Y)
for index, study in enumerate(train_studies):
    if study in gold.index:
        Y[index] = gold.loc[study, TARGETS].to_numpy(dtype=np.float32)
        W[index] = GOLD_WEIGHT
    elif study in report_labels.index:
        Y[index] = report_labels.loc[study, TARGETS].to_numpy(dtype=np.float32)
        confidence_columns = [target + "__conf" for target in TARGETS]
        confidence = report_labels.loc[study, confidence_columns].to_numpy(dtype=np.float32)
        W[index] = np.where(confidence > 0, 0.15 + 0.85 * confidence, 0.0)

usable = np.where((W.sum(axis=1) > 0) & (train_mask.sum(axis=1) > 0))[0]
log(f"usable studies={len(usable)}/{len(train_studies)}")

reports = train_df.set_index("StudyInstanceUID")["Report"].fillna("")


def grouped_multilabel_folds(indices: np.ndarray) -> np.ndarray:
    """Greedily balance targets while keeping duplicate normalized reports together."""
    grouped: dict[str, list[int]] = {}
    for index in indices:
        report = normalize_report(reports.get(train_studies[index], ""))
        key = hashlib.sha256((report or train_studies[index]).encode()).hexdigest()
        grouped.setdefault(key, []).append(int(index))

    known = W > 0
    positive = (Y >= 0.5) & known
    target_positive = positive[indices].sum(axis=0).astype(np.float64)
    target_size = len(indices) / N_FOLDS
    fold_size = np.zeros(N_FOLDS, dtype=np.float64)
    fold_positive = np.zeros((N_FOLDS, len(TARGETS)), dtype=np.float64)
    fold_of = np.full(len(train_studies), -1, dtype=np.int16)

    items = []
    for key, members in grouped.items():
        members_array = np.asarray(members, dtype=int)
        positives = positive[members_array].sum(axis=0).astype(np.float64)
        rarity = float(np.sum(positives / np.maximum(target_positive, 1.0)))
        items.append((key, members_array, positives, rarity))
    items.sort(key=lambda item: (-item[3], -len(item[1]), item[0]))

    rng = np.random.default_rng(RUN_SEED)
    fold_order = rng.permutation(N_FOLDS)
    for _, members, positives, _ in items:
        costs = []
        for fold in fold_order:
            candidate_size = fold_size.copy()
            candidate_positive = fold_positive.copy()
            candidate_size[fold] += len(members)
            candidate_positive[fold] += positives
            size_cost = np.mean((candidate_size / max(target_size, 1.0) - 1.0) ** 2)
            expected_positive = np.maximum(target_positive / N_FOLDS, 1.0)
            positive_cost = np.mean(
                (candidate_positive / expected_positive[None, :] - 1.0) ** 2
            )
            costs.append((positive_cost + 0.15 * size_cost, int(fold)))
        _, chosen = min(costs)
        fold_of[members] = chosen
        fold_size[chosen] += len(members)
        fold_positive[chosen] += positives
    return fold_of


fold_of = grouped_multilabel_folds(usable)
fold_records = pd.DataFrame(
    {
        "StudyInstanceUID": train_studies,
        "fold": fold_of,
        "usable": np.isin(np.arange(len(train_studies)), usable).astype(np.uint8),
    }
)
fold_records.to_csv("folds.csv", index=False)
fold_table = pd.DataFrame(
    {"fold": fold_of[usable], "gold": [train_studies[index] in gold.index for index in usable]}
).groupby("fold").agg(studies=("fold", "size"), gold=("gold", "sum"))
display(fold_table)


def write_submission(rank_sum: np.ndarray, n_models: int) -> pd.DataFrame:
    predicted = pd.DataFrame(rank_sum / max(n_models, 1), columns=TARGETS)
    predicted.insert(0, "StudyInstanceUID", test_studies)
    submission = test_df[["StudyInstanceUID"]].merge(predicted, on="StudyInstanceUID", how="left")
    submission[TARGETS] = submission[TARGETS].fillna(0.5)
    submission.to_csv("submission.csv", index=False)
    return submission


def train_fold(
    fold: int,
    train_indices: np.ndarray,
    valid_indices: np.ndarray,
    device: torch.device,
) -> tuple[dict[str, torch.Tensor], list[dict[str, float]]]:
    seed_everything(RUN_SEED + fold)
    model = build_model().to(device)
    optimizer = torch.optim.AdamW(
        [
            {"params": [p for p in model.backbone.parameters() if p.requires_grad], "lr": LR_BACKBONE},
            {"params": model.head.parameters(), "lr": LR_HEAD},
        ],
        weight_decay=WEIGHT_DECAY,
    )
    steps_per_epoch = max(1, len(train_indices) // BATCH_STUDIES)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=[LR_BACKBONE, LR_HEAD],
        total_steps=steps_per_epoch * EPOCHS,
        pct_start=0.15,
    )
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    valid_binary = (Y[valid_indices] >= 0.5).astype(np.int8)
    best_score, best_state, history = -np.inf, None, []

    for epoch in range(EPOCHS):
        model.train()
        shuffled = np.random.permutation(train_indices)
        losses = []
        for offset in range(0, steps_per_epoch * BATCH_STUDIES, BATCH_STUDIES):
            selection = shuffled[offset : offset + BATCH_STUDIES]
            group = int(np.random.randint(0, N_GROUPS))
            images = augment(take_group(train_cache, selection, group).to(device))
            batch_mask = torch.from_numpy(train_mask[selection]).to(device)
            labels = torch.from_numpy(Y[selection]).to(device)
            weights = torch.from_numpy(W[selection]).to(device)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
                logits = model(images, batch_mask)
                bce = F.binary_cross_entropy_with_logits(logits, labels, reduction="none")
                loss = (bce * weights).sum() / weights.sum().clamp_min(1.0)
                loss = loss + RANK_LOSS_WEIGHT * pairwise_rank_loss(logits.float(), labels, weights)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            losses.append(float(loss.detach().cpu()))

        valid_predictions = predict(model, train_cache, train_mask, valid_indices, device)
        score = macro_auc(valid_binary, valid_predictions)
        gold_indices = np.array(
            [index for index in valid_indices if train_studies[index] in gold.index], dtype=int
        )
        gold_score = float("nan")
        if len(gold_indices) >= 6:
            gold_y = gold.loc[[train_studies[index] for index in gold_indices], TARGETS].to_numpy(
                dtype=np.int8
            )
            gold_predictions = predict(model, train_cache, train_mask, gold_indices, device)
            gold_score = macro_auc(gold_y, gold_predictions)
        mean_loss = float(np.mean(losses))
        history.append(
            {"fold": fold, "epoch": epoch + 1, "loss": mean_loss,
             "weak_auc": score, "gold_auc": gold_score}
        )
        log(
            f"fold={fold} epoch={epoch + 1}/{EPOCHS} loss={mean_loss:.4f} "
            f"weak_auc={score:.4f} gold_auc={gold_score:.4f}"
        )
        if np.isfinite(score) and score > best_score:
            best_score = score
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        if time.time() - START_TIME > TRAIN_STOP_SECONDS:
            log("time budget reached during fold")
            break

    if best_state is None:
        best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
    del model, optimizer, scheduler, scaler
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return best_state, history

## Train the selected fold and export artifacts

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda" and not DEBUG:
    raise RuntimeError("Enable a Kaggle GPU accelerator before running the full notebook")

oof = np.full((len(train_studies), len(TARGETS)), np.nan, dtype=np.float32)
all_history = []

for fold in [TRAIN_FOLD]:
    train_indices = usable[fold_of[usable] != fold]
    valid_indices = usable[fold_of[usable] == fold]
    if len(train_indices) < BATCH_STUDIES or len(valid_indices) == 0:
        raise RuntimeError(
            f"fold {fold} is empty or too small: train={len(train_indices)} "
            f"valid={len(valid_indices)}"
        )
    log(f"starting fold={fold}: train={len(train_indices)} valid={len(valid_indices)}")
    fold_start = time.time()
    state, history = train_fold(fold, train_indices, valid_indices, device)
    all_history.extend(history)

    model = build_model().to(device)
    model.load_state_dict(state)
    oof[valid_indices] = predict(model, train_cache, train_mask, valid_indices, device)
    test_predictions = predict(
        model, test_cache, test_mask, np.arange(len(test_studies), dtype=int), device
    )
    test_ranks = pd.DataFrame(test_predictions).rank(pct=True).to_numpy()
    submission = write_submission(test_ranks, 1)
    artifact_stem = f"dinov2_seed{RUN_SEED}_fold{fold}"
    if SAVE_FOLD_MODELS:
        torch.save(
            {
                "model": state,
                "seed": RUN_SEED,
                "fold": fold,
                "targets": TARGETS,
                "config": {
                    "variant": "small",
                    "image_size": IMAGE_SIZE,
                    "crop_mm": CROP_MM,
                    "group_size": GROUP_SIZE,
                    "n_groups": N_GROUPS,
                    "slices": CACHE_SLICES,
                    "unfreeze_last": UNFREEZE_LAST,
                    "slots": [slot[0] for slot in SLOTS],
                    "fracture_pool": "max_probability",
                },
            },
            artifact_stem + ".pt",
        )
    np.save(artifact_stem + "_oof.npy", oof)
    np.save(artifact_stem + "_test.npy", test_predictions)
    pd.DataFrame(
        oof[valid_indices], columns=TARGETS, index=np.asarray(train_studies)[valid_indices]
    ).rename_axis("StudyInstanceUID").to_csv(artifact_stem + "_oof.csv")
    pd.DataFrame(
        test_predictions, columns=TARGETS, index=test_studies
    ).rename_axis("StudyInstanceUID").to_csv(artifact_stem + "_test.csv")
    del model, state, test_predictions
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

    fold_seconds = time.time() - fold_start
    log(f"finished seed={RUN_SEED} fold={fold} in {fold_seconds / 60:.1f} min")

history_df = pd.DataFrame(all_history)
history_df.to_csv(f"training_history_seed{RUN_SEED}_fold{TRAIN_FOLD}.csv", index=False)
gold_positions = np.array(
    [index for index, study in enumerate(train_studies) if study in gold.index], dtype=int
)
seen = gold_positions[np.isfinite(oof[gold_positions]).all(axis=1)]
if len(seen) >= 8:
    gold_y = gold.loc[[train_studies[index] for index in seen], TARGETS].to_numpy(dtype=np.int8)
    log(f"gold OOF macro AUC on {len(seen)} studies: {macro_auc(gold_y, oof[seen]):.4f}")

log(f"complete: submission.csv={submission.shape}, seed={RUN_SEED}, fold={TRAIN_FOLD}")
display(submission.head())

if DELETE_TEMP_CACHE:
    for cache_root in {train_cache.root, test_cache.root}:
        if cache_root.is_dir() and CACHE_ROOT in cache_root.parents:
            shutil.rmtree(cache_root)
    log("removed temporary compressed image caches")

## Next competitive experiments

Train folds 0-4 under one seed before adding more seeds. Package the resulting
checkpoints as a private Kaggle Dataset and use an inference-only notebook to rank
average them. Keep the fold table fixed for every architecture.

1. Improve report labels for the weakest gold-agreement targets.
2. Add DINOv2-base or a supervised ConvNeXt as a genuinely different encoder.
3. Increase slice groups only when the runtime and RAM measurements support it.
4. Rank-average out-of-fold-validated models; do not choose blends from public LB noise.